# Lesson 0008: the training loop as instrument

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/soroban/blob/main/lessons/0008-training-loop/lesson.ipynb)

Every lesson so far judged a model by its training loss and cheered when it fell. This one shows why that number, alone, lies. A model with more parameters than data points can drive its training loss to zero by memorizing, and a memorized answer key says nothing about unseen questions. The fix is a second number: the loss on held-out data. This notebook first fits three noisy points with a line and a parabola to see the gap by hand, then splits the TinyStories text and trains lesson 0007's transformer at three widths to watch the same gap open at scale. The full writeup is in the [lesson README](https://github.com/tamnd/soroban/tree/main/lessons/0008-training-loop).

![same points, two fits: the parabola memorizes and then misses](https://raw.githubusercontent.com/tamnd/soroban/main/lessons/0008-training-loop/assets/overfit.png)

## 0. Two fits to the same points

The true relationship is the line `y = 0.5x + 0.5`, sampled at `x = 0, 1, 2` with the middle point nudged off the line (noise), and two clean held-out points at `x = 3, 4`. A line has two parameters and keeps a small training error. A parabola has three parameters, one per training point, so it interpolates them exactly, training error zero, then bends away from the held-out points. Training error alone would pick the parabola, the worse model. The full page is [overfitting.md](https://github.com/tamnd/soroban/blob/main/maths/overfitting.md).

In [1]:
import numpy as np

TRAIN_X = np.array([0.0, 1.0, 2.0]); TRAIN_Y = np.array([0.5, 1.3, 1.5])
VAL_X = np.array([3.0, 4.0]);        VAL_Y = np.array([2.0, 2.5])

def mse(coef, x, y): return float(np.mean((np.polyval(coef, x) - y) ** 2))

line = np.polyfit(TRAIN_X, TRAIN_Y, 1)      # y = 0.5x + 0.6
parab = np.polyfit(TRAIN_X, TRAIN_Y, 2)     # y = -0.3x^2 + 1.1x + 0.5
print(f"line   train mse {mse(line, TRAIN_X, TRAIN_Y):.3f}   held-out mse {mse(line, VAL_X, VAL_Y):.3f}")
print(f"parab  train mse {mse(parab, TRAIN_X, TRAIN_Y):.3f}   held-out mse {mse(parab, VAL_X, VAL_Y):.3f}")
assert abs(mse(parab, TRAIN_X, TRAIN_Y)) < 1e-9        # memorized: train error zero
assert abs(mse(parab, VAL_X, VAL_Y) - 3.285) < 1e-9    # and 328x worse held out

line   train mse 0.020   held-out mse 0.010
parab  train mse 0.000   held-out mse 3.285


## 1. The threshold is parameters against data points

A polynomial with as many parameters as data points can pass through all of them, whatever the data: three parameters memorize three points. That ratio scales to the transformer. Lesson 0007's model has 58273 parameters and trained on 9311 characters, about six parameters per character, well past the point where memorizing is easy. That, not understanding, is why its training loss fell so far.

In [2]:
def param_count(V, D, T):
    return (V*D + T*D + 3*(D*D) + (D*D+D)
            + (D*4*D + 4*D) + (4*D*D + D) + 3*(2*D) + (D*V + V))

P = param_count(33, 64, 64)
print(f"{P} params on 9311 chars = {P/9311:.3f} params per char")
assert P == 58273

58273 params on 9311 chars = 6.259 params per char


## 2. The same instrument on the transformer

Split the text into 90 percent training and 10 percent held out, and train the same architecture at widths `D = 16, 32, 64` for three thousand steps each, measuring both losses every 150 steps. Wider models push training loss down and held-out loss up: the gap is overfitting. The held-out loss traces a U, bottoming early and then climbing; the bottom is where you should stop. (This cell needs torch, which Colab has preinstalled.)

![train loss keeps falling; val loss turns back up](https://raw.githubusercontent.com/tamnd/soroban/main/lessons/0008-training-loop/assets/gap.png)

In [3]:
CORPUS = """u don't have to be scared of the loud dog, i'll protect you . the mole felt so safe with the little girl. she was very kind and the mole soon came to trust her. he leaned against her and she kept him safe. the mole had found his best friend.
once upon a time, there was a wealthy man named tom. he had a big house near a cliff. tom liked to sort his many toys into different boxes. one sunny day, tom went outside to play with his toys. he took them all out of their boxes and spread them on the ground. he had fun playing with his cars, dolls, and balls. when it was time to go home, tom sorted his toys back into their boxes. he was happy to live in his big house near the cliff. and every day, he played with his toys and sorted them again and again.
once upon a time, there was a cool cat named tom. tom loved to go for a jog in the park. every day, he would put on his cool hat and go for a run. one sunny day, as tom was jogging, he saw a big tree. he decided to turn right and run around it. as he turned, he met a new friend, a dog named sam. sam was also going for a jog in the park. tom and sam jogged together every day. they would turn around the big tree, then sit under it to rest. they became best friends and had lots of fun in the cool park.
once upon a time, there was a dog named spot. spot was a very persistent dog. he loved to play and have fun. one day, spot heard his friends talking. we will celebrate! said one friend. spot was excited. he wanted to celebrate too. he ran to his friends and asked, can i celebrate too? his friends smiled and said, yes, spot! let's all celebrate together! they played games, ate yummy food, and laughed a lot. spot was very happy. his friends were happy too. they all had a great time celebrating. and they all lived happily ever after.
one day, a little girl named lily went to the park. she saw a pretty angel playing. the angel had big wings and a nice smile. lily wanted to catch the angel, but she was too slow. lily called out, angel, please wait for me! but the angel did not hear her. the angel was deaf. she could not hear anything. lily felt sad, but she had an idea. lily picked up a flower and threw it to the angel. the angel saw the flower and smiled at lily. she flew down to lily and they became friends. they played in the park all day, and lily was happy.
once upon a time, there was a long string. this string was very special. it lived in a big, pretty box. the string was very happy in the box. one day, a little boy found the box. he wanted to study the long string. he took the string out of the box and played with it. the string was very happy to be with the little boy. the little boy and the string became best friends. they played all day and had lots of fun. the long string was very happy to have a friend. and they lived happily ever after.
once upon a time, there was a polite bee named bob. bob lived in a big hive with all his bee friends. the hive was in a tall tree, near pretty flowers. bob loved his home. every day, bob and his friends went to the flowers to get food. they took the food back to the hive to store it. they worked together and shared with each other. one day, bob met a new friend, a butterfly named bella. bella was very nice and polite too. they played together and had lots of fun. from that day on, bob and bella were the best of friends.
once upon a time, there was a little beetle named bob. bob was very popular. all his friends liked him a lot. bob lived in a big green tree. one day, bob was playing with his friends when a new beetle appeared. the new beetle was shy and said, hi, i'm tim. can i play too? bob and his friends were happy to have a new friend. they all played together and had so much fun. tim was happy to be with bob and his friends. now, tim was popular too. they all lived happily in the big green tree.
once upon a time, there was a pretty flower. the flower lived in a big garden. one day, a little boy named tim saw the flower. he liked it a lot. tim said, i want to test if the flower can be mine. he picked the flower from the ground. but, oh no! the flower was spoiled. the pretty flower turned brown and sad. tim cried and told his mom, i picked the flower and it got spoiled. his mom said, you should not pick flowers, they are happy in the garden. tim learned that it is better to leave pretty things where they are.
once upon a time, there was a gifted bird named blue. blue could whistle the best songs in the forest. all the other animals loved to hear blue whistle. one day, blue found a shiny black rock. it was coal. blue took the coal to his friend, bunny. bunny liked the coal and used it to draw pictures on the ground. blue and bunny had a fun day playing with the coal and whistling songs. all the animals in the forest came to see the pictures and hear blue whistle. they all had a great time together.
once upon a time, there was an adorable little dog named max. max loved to play with his gear. he had a ball, a bone, and a rope. max played with his gear all day long. one day, max saw a big cat. the cat said, bow to me, little dog. max did not want to bow to the cat, but he did it anyway. the cat laughed and took max's gear. max went home without his gear. he was very sad. his owner tried to make him happy, but max missed his gear too much. the cat never gave max his gear back, and max stayed sad.
once upon a time, in a small house, there was a little girl named mia. mia had a pet bird named bob. bob lived in a cage. mia loved bob very much. one day, mia saw that bob was sad. she asked, bob, why are you sad? bob said, i want to be free and fly. mia felt sad for bob. she wanted to make bob happy. mia opened the cage door and let bob out. bob was very happy. he flew around the room. mia was proud of her bird. she said, bob, i love you! bob flew to mia and let her rub his head. they were both happy and played together all day.
one day, a popular cat named tom went for a walk. he saw a jar on the side of the road. tom was curious, so he stepped closer to look at it. hey, tom, said a bird named sue. what's in the jar? tom didn't know, so he opened the jar. out jumped a tiny frog! they were both surprised. thank you for letting me out, said the frog. i will give you a wish. tom and sue looked at each other. they wished to be friends forever. the frog smiled and their wish came true. they were all very happy.
one day, a boy named tim was very excited. his mom and dad were going to send him to the zoo. he had never been to the zoo before. tim could not wait to see all the animals. when they got to the zoo, tim saw a big lion. the lion roared loud. he also saw a tall giraffe with a long neck. the giraffe ate leaves from the tree. tim was so happy to see all the animals. at the end of the day, tim and his mom and dad went home. tim was very tired but still excited. he told all his friends about the zoo. tim could not wait to go back to the zoo again.
once upon a time, in a peaceful town, there was a big square. in the square, there were many kids who liked to play. they were very happy. one day, a little girl saw a puzzle on the ground. it had many square pieces. she wanted to solve it. so, she asked her friends to help her. they all worked together to solve the puzzle. they put the square pieces in the right place. soon, the puzzle was done. the kids were so happy and proud. they had a fun day in the peaceful town.
one day, a cat and a dog were in a park. the cat was comfortable on a bench. the dog was near a grill. the dog said, i want to play! the cat looked at the dog and said, okay, let's play! they played near the grill. the dog ran fast and the cat jumped high. they had fun. then, the dog's tail hit the grill. ouch! said the dog. the cat ran to help. the cat gave the dog a soft slap on the back. the dog felt better. they went back to play and had a great day.
one day, tim went for a walk with his mom. they saw a big pile of junk near their house. tim was very alert and observed something shiny in the junk. mom, look! tim said. i see something shiny. can i get it? his mom said, okay, but be careful. tim carefully moved the junk and found a toy car. he was so happy. he showed the car to his mom, and she smiled. from then on, tim always observed his surroundings and found many more treasures. he learned that being alert can lead to finding special things.
once upon a time, there was a little boat. the boat liked to go to the shore. one day, the boat saw a big load. the load was heavy and uncomfortable. the boat wanted to help. so, the boat took the load to the shore. the load made the boat very uncomfortable. the boat felt slow and tired. in the end, the boat could not carry the load anymore. the boat stopped moving and stayed on the shore. the boat was sad and uncomfortable forever.
once upon a time, a kind girl named lily went for a walk. she saw a pretty purse on the ground. lily liked the purse and wanted to find who it belonged to. lily asked her friends, do you know who lost this purse? her friends did not know, but they all admired the purse. they thought it was very nice. finally, lily found a sad lady who had lost her purse. the lady was so happy when lily gave it back to her. the lady said, thank you, kind girl! lily smiled and felt good for being helpful."""

try:
    import math, torch, torch.nn.functional as F

    chars = sorted(set(CORPUS)); V = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    full = torch.tensor([stoi[c] for c in CORPUS], dtype=torch.long)
    cut = int(0.9 * len(full))
    train_data, val_data = full[:cut], full[cut:]
    B, T = 32, 64
    print(f"split {len(train_data)} train chars, {len(val_data)} val chars, vocab {V}")

    def build(D):
        class Block(torch.nn.Module):
            def __init__(self):
                super().__init__()
                self.tok = torch.nn.Embedding(V, D); self.pos = torch.nn.Embedding(T, D)
                self.q = torch.nn.Linear(D, D, bias=False)
                self.k = torch.nn.Linear(D, D, bias=False)
                self.v = torch.nn.Linear(D, D, bias=False)
                self.proj = torch.nn.Linear(D, D)
                self.ln1 = torch.nn.LayerNorm(D); self.ln2 = torch.nn.LayerNorm(D)
                self.mlp = torch.nn.Sequential(
                    torch.nn.Linear(D, 4*D), torch.nn.GELU(), torch.nn.Linear(4*D, D))
                self.lnf = torch.nn.LayerNorm(D); self.head = torch.nn.Linear(D, V)
                self.register_buffer("mask", torch.tril(torch.ones(T, T)))
            def forward(self, idx):
                Tt = idx.shape[1]
                x = self.tok(idx) + self.pos(torch.arange(Tt)); h = self.ln1(x)
                att = (self.q(h) @ self.k(h).transpose(-2, -1)) / math.sqrt(D)
                att = att.masked_fill(self.mask[:Tt, :Tt] == 0, float("-inf"))
                att = F.softmax(att, dim=-1)
                x = x + self.proj(att @ self.v(h)); x = x + self.mlp(self.ln2(x))
                return self.head(self.lnf(x))
        return Block()

    def get_batch(data, g):
        ix = torch.randint(0, len(data)-T-1, (B,), generator=g)
        return (torch.stack([data[i:i+T] for i in ix]),
                torch.stack([data[i+1:i+T+1] for i in ix]))

    def run(D):
        torch.manual_seed(1337); model = build(D)
        opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
        g = torch.Generator().manual_seed(1337)
        np_ = sum(p.numel() for p in model.parameters())
        @torch.no_grad()
        def est(data):
            ge = torch.Generator().manual_seed(0)
            losses = []
            for _ in range(50):
                x, y = get_batch(data, ge)
                losses.append(F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item())
            return sum(losses) / len(losses)
        best, bs, ft, fv = 1e9, 0, 0.0, 0.0
        for step in range(3001):
            if step % 150 == 0:
                tl, vl = est(train_data), est(val_data)
                if vl < best: best, bs = vl, step
                if step == 3000: ft, fv = tl, vl
            if step < 3000:
                x, y = get_batch(train_data, g)
                loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
                opt.zero_grad(); loss.backward(); opt.step()
        print(f"  D={D:2d}  {np_:5d} params  final train {ft:.3f}  final val {fv:.3f}  best val {best:.3f} @ step {bs}")
        return ft, fv, best

    rows = [run(D) for D in (16, 32, 64)]
    assert rows[0][0] > rows[2][0]      # more params -> lower training loss
    assert rows[0][1] < rows[2][1]      # more params -> higher held-out loss
    assert rows[2][2] < 2.1651 < rows[2][1]   # early-stopped beats bigram; overtrained loses
    print("early-stopped D=64 beats the bigram 2.1651; trained to the end it loses to it")
except ImportError:
    print("torch not installed, skipping the training cell (Colab has it preinstalled)")

split 8379 train chars, 932 val chars, vocab 33


  D=16   5377 params  final train 1.286  final val 2.096  best val 1.978 @ step 1500


  D=32  16865 params  final train 0.932  final val 2.323  best val 1.964 @ step 1050


  D=64  58273 params  final train 0.323  final val 4.136  best val 1.880 @ step 600
early-stopped D=64 beats the bigram 2.1651; trained to the end it loses to it


## Exercises

1. Which width has the lowest training loss at step 3000, and which has the lowest held-out loss? They are different models. Predict, then read the printout.
2. Recompute the parabola's held-out error by hand: evaluate `y = -0.3x^2 + 1.1x + 0.5` at `x = 3, 4`, subtract the true `2.0, 2.5`, square, average. You should get 3.285.
3. The held-out minimum arrives at step 1500 for `D = 16` and step 600 for `D = 64`. Why does more capacity make it arrive sooner?

Worked answers are in the [lesson README](https://github.com/tamnd/soroban/tree/main/lessons/0008-training-loop) and asserted in `train.py`. Lesson 0009 turns from what a model learns to what it costs to make it learn: the FLOPs ledger.